<a href="https://colab.research.google.com/github/M4rck0/Procesamiento_Clasificacion_Datos/blob/main/Tarea_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Librerías
import kagglehub
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Funciones

# Clasificación
def clasificar_sentimiento(rating):
    if rating <= 2:
        return "negativo"
    elif rating == 3:
        return "neutral"
    else:
        return "positivo"

In [3]:
# Descargar Kaggle
ruta = kagglehub.dataset_download("dongrelaxman/amazon-reviews-dataset")

Using Colab cache for faster access to the 'amazon-reviews-dataset' dataset.


In [4]:
# Archivo
os.listdir(ruta)

['Amazon_Reviews.csv']

In [5]:
# Leer base
ruta_archivo = os.path.join(ruta, "Amazon_Reviews.csv")
df_amazon = pd.read_csv(ruta_archivo, engine = "python")

In [6]:
# Selección columnas
df_amazon = df_amazon[["Country", "Review Count", "Review Date", "Rating", "Review Title", "Review Text"]].copy()

# Eliminar filas sin reseña
df_amazon = df_amazon.dropna(subset=["Rating", "Review Text"])

# Extraer número de estrellas (rating)
df_amazon["rating"] = df_amazon["Rating"].str.extract(r"Rated (\d+) out of 5 stars").astype(float)

# Eliminar ratings que no se pudieron convertir
df_amazon = df_amazon.dropna(subset=["rating"])

df_amazon.head()

,Country,Review Count,Review Date,Rating,Review Title,Review Text,rating
0,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...",1.0
1,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,1.0
2,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,1.0
3,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,1.0
4,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,1.0


In [8]:
df_amazon["sentimiento"] = df_amazon["rating"].apply(clasificar_sentimiento)

df_amazon[["Review Text", "rating", "sentimiento"]].head()

,Review Text,rating,sentimiento
0,"I registered on the website, tried to order a ...",1.0,negativo
1,Had multiple orders one turned up and driver h...,1.0,negativo
2,I informed these reprobates that I WOULD NOT B...,1.0,negativo
3,I have bought from Amazon before and no proble...,1.0,negativo
4,If I could give a lower rate I would! I cancel...,1.0,negativo


In [9]:
df_amazon["sentimiento"].value_counts()

,count
sentimiento,
negativo,14350
positivo,5820
neutral,885


In [10]:
df_amazon["sentimiento"].value_counts(normalize=True)

,proportion
sentimiento,
negativo,0.681548
positivo,0.276419
neutral,0.042033


In [11]:
df_amazon["rating"].value_counts().sort_index()

,count
rating,
1.0,13123
2.0,1227
3.0,885
4.0,1292
5.0,4528


In [13]:
# Columna texto
df_amazon["texto"] = (df_amazon["Review Title"].fillna("") + " " + df_amazon["Review Text"].fillna(""))
df_amazon[["texto", "rating", "sentimiento"]].head()

,texto,rating,sentimiento
0,A Store That Doesn't Want to Sell Anything I r...,1.0,negativo
1,Had multiple orders one turned up and… Had mul...,1.0,negativo
2,I informed these reprobates I informed these r...,1.0,negativo
3,Advertise one price then increase it on websit...,1.0,negativo
4,If I could give a lower rate I would If I coul...,1.0,negativo


In [14]:
vectorizador = TfidfVectorizer(lowercase=True, stop_words="english", max_features=5000) # Mínusculas, inglés y máximo 5,000 palabras

X = vectorizador.fit_transform(df_amazon["texto"])
y = df_amazon["sentimiento"]

print(X.shape)

(21055, 5000)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Proporciones similares

modelo = LogisticRegression(max_iter=1000, class_weight="balanced")

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negativo       0.95      0.89      0.92      2870
     neutral       0.17      0.40      0.23       177
    positivo       0.88      0.85      0.87      1164

    accuracy                           0.85      4211
   macro avg       0.67      0.71      0.67      4211
weighted avg       0.90      0.85      0.87      4211



In [16]:
print(confusion_matrix(y_test, y_pred))

[[2541  242   87]
 [  64   70   43]
 [  63  112  989]]


Se utilizó un conjunto de datos de reseñas de Amazon obtenido de Kaggle. El dataset contiene información del país, fecha de reseña, calificación, título y texto de la reseña.

Para construir la variable objetivo, se extrajo la calificación numérica de la columna Rating. Posteriormente, las reseñas se clasificaron en tres categorías de sentimiento: negativo para calificaciones de 1 y 2 estrellas, neutral para 3 estrellas, y positivo para 4 y 5 estrellas.

El texto utilizado para el modelo se construyó uniendo el título de la reseña y el contenido de la reseña. Después, se aplicó una vectorización TF-IDF con un máximo de 5,000 términos, eliminando palabras vacías en inglés.

Finalmente, se entrenó una regresión logística con balanceo de clases. El modelo obtuvo una exactitud general de 85%. El desempeño fue alto para las clases negativa y positiva, con F1-score de 0.92 y 0.87 respectivamente. Sin embargo, la clase neutral tuvo un F1-score bajo de 0.23, lo cual se explica por el fuerte desbalance de clases, ya que las reseñas neutrales representan solamente el 4.2% del total.